In [1]:
from google.colab import files
uploaded = files.upload()

Saving retail_transactions_100k.csv to retail_transactions_100k.csv


In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("RetailDataAnalysis") \
    .getOrCreate()

In [4]:
df = spark.read.csv(
    "retail_transactions_100k.csv",
    header=True,
    inferSchema=True
)

In [5]:
df.show(5)

+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+---------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|   product_name|   category|quantity|unit_price|discount_pct|payment_method|   status|rating|
+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+---------+------+
| O000001|2026-02-27|     C00205|Customer_00205|     Pune|  West|      P004|27 Inch Monitor|Electronics|       2|  16961.87|           0|           COD|Delivered|  NULL|
| O000002|2026-04-22|     C01906|Customer_01906|  Lucknow| North|      P010|     Web Camera|Accessories|       1|   4241.16|          20|   Net Banking|Delivered|     1|
| O000003|2026-03-23|     C03463|Customer_03463|Ahmedabad|  West|      P005|   Office Chair|  Furniture|       2|   8590.05|          10|           UP

In [6]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- rating: integer (nullable = true)



In [7]:
print("Total records:", df.count())

Total records: 100000


In [8]:
# 1. Display the first 20 records
df.show(20)

# 2. Display the schema
df.printSchema()

# 3. Display all column names
print(df.columns)

# 4. Count the total number of records
df.count()

# 5. Display specific columns (order ID, product name, city, quantity, unit price)
df.select("order_id", "product_name", "city", "quantity", "unit_price").show()

# 6. Display only Electronics transactions
df.filter(df.category == "Electronics").show()

# 7. Display transactions from Hyderabad
df.filter(df.city == "Hyderabad").show()

# 8. Display transactions where quantity is greater than 3
df.filter(df.quantity > 3).show()

# 9. Display transactions where unit price is greater than 20,000
df.filter(df.unit_price > 20000).show()

# 10. Display Delivered transactions
df.filter(df.status == "Delivered").show()

+--------+----------+-----------+--------------+---------+------+----------+-------------------+-----------+--------+----------+------------+--------------+----------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|       product_name|   category|quantity|unit_price|discount_pct|payment_method|    status|rating|
+--------+----------+-----------+--------------+---------+------+----------+-------------------+-----------+--------+----------+------------+--------------+----------+------+
| O000001|2026-02-27|     C00205|Customer_00205|     Pune|  West|      P004|    27 Inch Monitor|Electronics|       2|  16961.87|           0|           COD| Delivered|  NULL|
| O000002|2026-04-22|     C01906|Customer_01906|  Lucknow| North|      P010|         Web Camera|Accessories|       1|   4241.16|          20|   Net Banking| Delivered|     1|
| O000003|2026-03-23|     C03463|Customer_03463|Ahmedabad|  West|      P005|       Office Chair|  Furniture|       2|   8590.

In [9]:
from pyspark.sql.functions import col

# 11. Find Electronics transactions from Hyderabad
df.filter((col("category") == "Electronics") & (col("city") == "Hyderabad")).show()

# 12. Find Delivered transactions from the South region
df.filter((col("status") == "Delivered") & (col("region") == "South")).show()

# 13. Find orders having quantity greater than 2 and unit price greater than 10,000
df.filter((col("quantity") > 2) & (col("unit_price") > 10000)).show()

# 14. Find transactions from either Mumbai or Pune
df.filter(col("city").isin(["Mumbai", "Pune"])).show()

# 15. Find transactions that are not Cancelled
df.filter(col("status") != "Cancelled").show()

# 16. Find Furniture products costing more than 8,000
df.filter((col("category") == "Furniture") & (col("unit_price") > 8000)).show()

# 17. Find Accessories products with quantity greater than 3
df.filter((col("category") == "Accessories") & (col("quantity") > 3)).show()

+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+----------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|   product_name|   category|quantity|unit_price|discount_pct|payment_method|    status|rating|
+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+----------+------+
| O000015|2026-06-07|     C01962|Customer_01962|Hyderabad| South|      P004|27 Inch Monitor|Electronics|       5|  19287.21|           0|   Net Banking|Processing|  NULL|
| O000032|2026-08-05|     C02267|Customer_02267|Hyderabad| South|      P001|     Laptop Pro|Electronics|       3|  67819.64|           5|   Credit Card|Processing|  NULL|
| O000063|2026-05-22|     C04554|Customer_04554|Hyderabad| South|      P009|    Smart Watch|Electronics|       2|  11204.38|          10|   Net B

In [10]:
# 18. Select only order_id, customer_name, product_name, unit_price
df.select("order_id", "customer_name", "product_name", "unit_price").show()

# 19 & 20. Rename columns (unit_price -> price, customer_name -> customer)
df_renamed = df.withColumnRenamed("unit_price", "price") \
               .withColumnRenamed("customer_name", "customer")

# 21 & 22. Drop columns
df_dropped_rating = df.drop("rating")
df_new_dropped = df.drop("customer_name", "rating", "payment_method")

# 23. Add gross_amount (quantity * unit_price)
df = df.withColumn("gross_amount", col("quantity") * col("unit_price"))

# 24. Add discount_amount (gross_amount * discount_pct / 100)
df = df.withColumn("discount_amount", col("gross_amount") * (col("discount_pct") / 100))

# 25. Add final_amount (gross_amount - discount_amount)
df = df.withColumn("final_amount", col("gross_amount") - col("discount_amount"))

# 26. Add constant column country = 'India'
from pyspark.sql.functions import lit
df = df.withColumn("country", lit("India"))

# 27. Add high_value (Yes if final_amount >= 50000 else No)
from pyspark.sql.functions import when
df = df.withColumn("high_value", when(col("final_amount") >= 50000, "Yes").otherwise("No"))

+--------+--------------+-------------------+----------+
|order_id| customer_name|       product_name|unit_price|
+--------+--------------+-------------------+----------+
| O000001|Customer_00205|    27 Inch Monitor|  16961.87|
| O000002|Customer_01906|         Web Camera|   4241.16|
| O000003|Customer_03463|       Office Chair|   8590.05|
| O000004|Customer_00356|        Smart Watch|  12908.38|
| O000005|Customer_01576|         Laptop Pro|  67839.91|
| O000006|Customer_02989|         Study Desk|   13349.3|
| O000007|Customer_01339|             Tablet|  30193.06|
| O000008|Customer_01877|         Study Desk|  13479.74|
| O000009|Customer_03242|Mechanical Keyboard|   4240.53|
| O000010|Customer_01797|        Smart Watch|  11214.55|
| O000011|Customer_03153|         Web Camera|   4219.56|
| O000012|Customer_02186|     Wireless Mouse|   1484.34|
| O000013|Customer_04101|        Smart Watch|   12711.4|
| O000014|Customer_04419|         Laptop Pro|  63171.22|
| O000015|Customer_01962|    27

In [11]:
# 28. Create order_size (High, Medium, Low)
df = df.withColumn("order_size",
    when(col("final_amount") > 50000, "High")
    .when(col("final_amount") >= 15000, "Medium")
    .otherwise("Low")
)

# 29. Create quantity_type (Bulk if quantity >= 4 else Regular)
df = df.withColumn("quantity_type", when(col("quantity") >= 4, "Bulk").otherwise("Regular"))

# 30. Create delivery-status categories if needed based on your status values
df = df.withColumn("delivery_category",
    when(col("status") == "Delivered", "Completed")
    .when(col("status").isin(["Cancelled", "Returned"]), "Failed")
    .otherwise("In Progress")
)

In [12]:
# 31. Sort by unit price highest to lowest
df.orderBy(col("unit_price").desc()).show()

# 33. Sort alphabetically by city
df.orderBy(col("city").asc()).show()

# 35. Display 10 highest-value transactions
df.orderBy(col("final_amount").desc()).limit(10).show()

# 36. Display all unique cities[cite: 6]
df.select("city").distinct().show()

# 40. Count unique customers[cite: 6]
df.select("customer_id").distinct().count()

# 42-48. Aggregations
from pyspark.sql.functions import sum, avg, min, max
df.select(
    sum("quantity").alias("total_quantity"),
    sum("gross_amount").alias("total_gross_sales"),
    avg("unit_price").alias("avg_unit_price"),
    max("unit_price").alias("max_unit_price"),
    min("unit_price").alias("min_unit_price"),
    sum("discount_amount").alias("total_discount"),
    sum("final_amount").alias("total_final_sales")
).show()

+--------+----------+-----------+--------------+---------+------+----------+------------+-----------+--------+----------+------------+--------------+----------+------+------------------+------------------+------------------+-------+----------+----------+-------------+-----------------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|product_name|   category|quantity|unit_price|discount_pct|payment_method|    status|rating|      gross_amount|   discount_amount|      final_amount|country|high_value|order_size|quantity_type|delivery_category|
+--------+----------+-----------+--------------+---------+------+----------+------------+-----------+--------+----------+------------+--------------+----------+------+------------------+------------------+------------------+-------+----------+----------+-------------+-----------------+
| O016427|2026-01-20|     C01275|Customer_01275|Ahmedabad|  West|      P001|  Laptop Pro|Electronics|       1|  70199.47|           0|     

In [13]:
# 49. Count transactions by status
df.groupBy("status").count().show()

# 52. Calculate total sales by region
df.groupBy("region").sum("final_amount").show()

# 62. Find the top 5 cities based on total sales
df.groupBy("city") \
  .sum("final_amount") \
  .withColumnRenamed("sum(final_amount)", "total_sales") \
  .orderBy(col("total_sales").desc()) \
  .limit(5).show()

# 65-67. Date processing (extract year, month, day)
from pyspark.sql.functions import year, month, dayofmonth
df = df.withColumn("year", year("order_date")) \
       .withColumn("month", month("order_date")) \
       .withColumn("day", dayofmonth("order_date"))

# 73-76. Check for missing values
df.filter(col("payment_method").isNull()).show()
print("Missing payment methods count:", df.filter(col("payment_method").isNull()).count())

# 77 & 78. Replace missing values
df = df.na.fill({"payment_method": "Unknown", "rating": 0})

# 79. Drop records having missing payment methods
df = df.dropna(subset=["payment_method"])

+----------+-----+
|    status|count|
+----------+-----+
|  Returned| 5912|
|   Shipped|13950|
|Processing|12115|
| Cancelled| 9927|
| Delivered|58096|
+----------+-----+

+------+--------------------+
|region|   sum(final_amount)|
+------+--------------------+
| South|1.2736728779755006E9|
|  East| 4.188441210040009E8|
|  West|1.2745074508245018E9|
| North|1.2742773118394942E9|
+------+--------------------+

+---------+--------------------+
|     city|         total_sales|
+---------+--------------------+
|     Pune| 4.380059195804999E8|
|Ahmedabad|4.2995853770049965E8|
|Hyderabad| 4.282570748114998E8|
|  Lucknow|4.2674751151249886E8|
|Bangalore|4.2477450649949884E8|
+---------+--------------------+

+--------+----------+-----------+--------------+---------+------+----------+-------------------+-----------+--------+----------+------------+--------------+----------+------+------------------+------------------+------------------+-------+----------+----------+-------------+--------------